# Pepagora Category Mapper — Elasticsearch Approach
Step-by-step notebook to ingest and query product data using Elasticsearch.

## Step 1 — Install Dependencies
Install `elasticsearch` (official Python client) and `pandas` for data loading. We also install `ftfy` — a battle-tested library for fixing broken/mojibake Unicode text.

In [14]:
%pip install "elasticsearch>=8.0,<9.0" pandas ftfy tqdm --quiet
import importlib.metadata
print(f"elasticsearch version: {importlib.metadata.version('elasticsearch')}")

Note: you may need to restart the kernel to use updated packages.
elasticsearch version: 8.19.3


## Step 2 — Connect to Elasticsearch & Verify

In [15]:
from elasticsearch import Elasticsearch, ConnectionError as ESConnectionError

ES_HOST = "http://localhost:9200"

es = Elasticsearch(
    ES_HOST,
    request_timeout=10,
    retry_on_timeout=True,
)

# In elasticsearch-py 9.x ping() is removed — use es.info() directly
try:
    info   = es.info()
    health = es.cluster.health()
    print(f"✅ Connected to Elasticsearch")
    print(f"   Host    : {ES_HOST}")
    print(f"   Cluster : {info['cluster_name']}")
    print(f"   Version : {info['version']['number']}")
    print(f"   Status  : {health['status'].upper()}")
    print(f"   Nodes   : {health['number_of_nodes']}")
except ESConnectionError as exc:
    raise ConnectionError(f"❌ Cannot reach Elasticsearch at {ES_HOST}\n{exc}")

✅ Connected to Elasticsearch
   Host    : http://localhost:9200
   Cluster : docker-cluster
   Version : 8.12.0
   Status  : GREEN
   Nodes   : 1


## Step 3 — Load CSV Data
The file has **100,000 rows**. We read it with `utf-8` encoding first (no BOM detected). We also rename the dot-notation columns to flat names for easier access.

In [16]:
import pandas as pd

CSV_PATH = r"Dataset/pepagoraDb.liveproducts.csv"

df = pd.read_csv(
    CSV_PATH,
    encoding="utf-8",          # file is UTF-8 without BOM
    on_bad_lines="warn",        # warn on malformed rows instead of crashing
    dtype=str,                  # read everything as string — avoids mixed-type issues
)

# Flatten dot-notation column names  →  category_name, subCategory_name, etc.
df.columns = [c.replace(".", "_") for c in df.columns]

print(f"Loaded {len(df):,} rows  ×  {len(df.columns)} columns")
print("Columns:", list(df.columns))

Loaded 100,000 rows  ×  8 columns
Columns: ['_id', 'productName', 'productDescription', 'category_name', 'subCategory_name', 'productCategory__id', 'productCategory_name', 'productCategory_uniqueId']


## Step 4 — Detect & Fix UTF-8 Encoding Issues
Common problems in this dataset:
| Broken text | Cause | Fixed |
|---|---|---|
| `Â°C` | `°` (U+00B0) read as Latin-1 | `°C` |
| `â€™` | `'` (U+2019 curly apostrophe) mojibake | `'` |
| `â€œ` / `â€` | `"` / `"` curly double-quotes mojibake | `"` / `"` |

`ftfy.fix_text()` detects and corrects all of these automatically. After fixing, we also **normalise** smart quotes to plain ASCII equivalents so Elasticsearch tokenisation is consistent.

In [17]:
import ftfy

# Characters to normalise after ftfy fixes mojibake
QUOTE_MAP = str.maketrans({
    "\u2018": "'",   # LEFT  SINGLE QUOTATION MARK  '
    "\u2019": "'",   # RIGHT SINGLE QUOTATION MARK  '
    "\u201c": '"',   # LEFT  DOUBLE QUOTATION MARK  "
    "\u201d": '"',   # RIGHT DOUBLE QUOTATION MARK  "
    "\u2013": "-",   # EN DASH
    "\u2014": "-",   # EM DASH
    "\u00b0": "°",   # DEGREE SIGN — keep as-is (valid UTF-8, just ensure no mojibake)
})

TEXT_COLUMNS = ["productName", "productDescription", "category_name",
                "subCategory_name", "productCategory_name"]

def clean_text(value: str) -> str:
    if not isinstance(value, str):
        return value
    fixed = ftfy.fix_text(value)        # repair mojibake (e.g. Â° → °, â€™ → ')
    fixed = fixed.translate(QUOTE_MAP)  # normalise smart quotes → plain ASCII
    return fixed.strip()

# --- Scan: show sample broken rows BEFORE fixing ---
print("=== Sample rows with suspected encoding issues (before fix) ===")
mask = df["productDescription"].str.contains(r"Â|â€|Ã", regex=True, na=False)
broken = df.loc[mask, ["productName", "productDescription"]].head(3)
if broken.empty:
    print("No obvious mojibake detected — data may already be clean.")
else:
    for _, row in broken.iterrows():
        print(f"\n[BEFORE] {row['productName'][:80]}")
        print(f"         {row['productDescription'][:120]}")

# --- Fix all text columns ---
for col in TEXT_COLUMNS:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# --- Verify: same rows after fix ---
print("\n=== Same rows AFTER fix ===")
if not broken.empty:
    for idx in broken.index:
        print(f"\n[AFTER]  {df.at[idx, 'productName'][:80]}")
        print(f"         {df.at[idx, 'productDescription'][:120]}")
else:
    print("Nothing to verify — no broken rows found.")


=== Sample rows with suspected encoding issues (before fix) ===
No obvious mojibake detected — data may already be clean.

=== Same rows AFTER fix ===
Nothing to verify — no broken rows found.


In [18]:
# Quick sanity check — confirm no residual mojibake remains
remaining = df[TEXT_COLUMNS].apply(
    lambda col: col.str.contains(r"Â|â€|Ã", regex=True, na=False)
).any(axis=1).sum()

print(f"Rows with residual encoding issues after fix: {remaining}")
print(f"\nDataset ready — {len(df):,} clean records.")
df[["productName", "category_name", "subCategory_name", "productCategory_name"]].head(5)

Rows with residual encoding issues after fix: 0

Dataset ready — 100,000 clean records.


,productName,category_name,subCategory_name,productCategory_name
0,Industrial Power Systems Medium Voltage Capaci...,Electronics & Electrical,"Capacitors, Resistors, Inductors",Electrolytic Capacitors
1,Non Explosive Demolition Agent Low Heat Expans...,Raw Materials & Chemicals,Cosmetic Raw Materials,Surfactants
2,ISO Certified Medium Grain White Rice Rava And...,Food & Agriculture,"Grains, Cereals & Pulses",Rice
3,Premium Andhra Sona Masoori Medium Grain White...,Food & Agriculture,"Grains, Cereals & Pulses",Rice
4,Premium Executive Roll On Ball Pens With Metal...,Office Supplies & Equipment,Office Stationery,"Pens (Ballpoint, Gel, Fountain)"


## Step 5 — Create Index with Optimal Mapping
We create `pepagora_products` with:
- **Custom `product_english` analyzer** — standard tokenizer + lowercase + stop words + English stemmer
- **`productName`** — `text` (analyzed) + `.keyword` (exact/aggregations) + `.suggest` (autocomplete-ready)
- **`productDescription`** — `text` (analyzed, supporting context)
- **`category_name`, `subCategory_name`, `productCategory_name`** — `keyword` (for filters & aggregations) + `.text` (for fuzzy search)
- **Write-optimised settings** — replicas off, refresh disabled during ingest, restored after

In [19]:
INDEX_NAME = "pepagora_products"

INDEX_CONFIG = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,          # single-node — no replicas needed
        "refresh_interval": "1s",         # normal read setting (changed during ingest)
        "analysis": {
            "filter": {
                "english_stop": {
                    "type": "stop",
                    "stopwords": "_english_"
                },
                "english_stemmer": {
                    "type": "stemmer",
                    "language": "english"
                },
                "edge_ngram_filter": {
                    "type": "edge_ngram",
                    "min_gram": 2,
                    "max_gram": 20,
                    "token_chars": ["letter", "digit"]
                }
            },
            "analyzer": {
                "product_english": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": [
                        "lowercase",
                        "english_stop",
                        "english_stemmer"
                    ]
                },
                "product_autocomplete": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "edge_ngram_filter"]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            # ── Search fields ─────────────────────────────────────────────────
            "productName": {
                "type": "text",
                "analyzer": "product_english",
                "fields": {
                    "keyword": {                       # exact match & aggregations
                        "type": "keyword",
                        "ignore_above": 512
                    },
                    "suggest": {                       # autocomplete — future-ready
                        "type": "completion"
                    },
                    "ngram": {                         # edge-ngram: fast mid-word & prefix
                        "type": "text",
                        "analyzer": "product_autocomplete",
                        "search_analyzer": "standard"
                    }
                }
            },
            "productDescription": {
                "type": "text",
                "analyzer": "product_english"
            },

            # ── Category fields ───────────────────────────────────────────────
            "category_name": {
                "type": "keyword",                     # filters & aggregations
                "fields": {
                    "text": {                          # fuzzy text search on category
                        "type": "text",
                        "analyzer": "standard"
                    }
                }
            },
            "subCategory_name": {
                "type": "keyword",
                "fields": {
                    "text": { "type": "text", "analyzer": "standard" }
                }
            },
            "productCategory_name": {
                "type": "keyword",
                "fields": {
                    "text": { "type": "text", "analyzer": "standard" }
                }
            }
            # ❌ productCategory__id       — not indexed
            # ❌ productCategory_uniqueId  — not indexed
        }
    }
}

# ── Delete existing index if it exists, then create fresh ────────────────────
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f"🗑️  Deleted existing index '{INDEX_NAME}'")

es.indices.create(index=INDEX_NAME, **INDEX_CONFIG)
print(f"✅ Index '{INDEX_NAME}' created successfully")

# Verify
mapping = es.indices.get_mapping(index=INDEX_NAME)
fields  = list(mapping[INDEX_NAME]["mappings"]["properties"].keys())
print(f"   Fields  : {fields}")

settings = es.indices.get_settings(index=INDEX_NAME)
shards   = settings[INDEX_NAME]["settings"]["index"]["number_of_shards"]
print(f"   Shards  : {shards}")
print(f"   Analyzer: product_english (standard + lowercase + stop + english stemmer)")

🗑️  Deleted existing index 'pepagora_products'
✅ Index 'pepagora_products' created successfully
   Fields  : ['category_name', 'productCategory_name', 'productDescription', 'productName', 'subCategory_name']
   Shards  : 1
   Analyzer: product_english (standard + lowercase + stop + english stemmer)


## Step 5b -- Re-index with Edge N-gram Support
The mapping now includes `edge_ngram_filter` and `product_autocomplete` analyzer.
Run this cell to **drop and recreate** the index, then re-run **Step 6** to ingest all 100k products.

| Analyzer | Used at | Purpose |
|---|---|---|
| `product_autocomplete` | **index time** | Splits `"bath towels"` into `[ba, bat, bath, to, tow, towel, towels]` |
| `standard` | **search time** | Keeps query tokens whole: `[bath, towels]` |

This asymmetry is intentional -- build big at index time, query fast at search time.

In [21]:
# Step 5b: Drop old index and recreate with edge-ngram mapping
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f'Deleted existing index: {INDEX_NAME}')

es.indices.create(index=INDEX_NAME, **INDEX_CONFIG)
print(f'Index {INDEX_NAME!r} recreated with edge-ngram support')

# Verify the new analyzer is registered
analyze_resp = es.indices.analyze(
    index=INDEX_NAME,
    body={"analyzer": "product_autocomplete", "text": "bath towels"}
)
tokens = [t['token'] for t in analyze_resp['tokens']]
print(f'Edge-ngram tokens for "bath towels": {tokens}')
print()
print('NOTE: Now re-run Step 6 (bulk ingest cell) to repopulate the index.')

Deleted existing index: pepagora_products
Index 'pepagora_products' recreated with edge-ngram support
Edge-ngram tokens for "bath towels": ['ba', 'bat', 'bath', 'to', 'tow', 'towe', 'towel', 'towels']

NOTE: Now re-run Step 6 (bulk ingest cell) to repopulate the index.


# Step 6 bulk ingest code 

In [22]:
import time
from tqdm.notebook import tqdm
from elasticsearch.helpers import streaming_bulk

CHUNK_SIZE = 500   # docs per bulk request

# ── Speed up ingest: disable auto-refresh ────────────────────────────────────
es.indices.put_settings(
    index=INDEX_NAME,
    body={"index": {"refresh_interval": "-1"}}
)

# ── Build action generator ────────────────────────────────────────────────────
FIELDS = [
    "productName", "productDescription",
    "category_name", "subCategory_name", "productCategory_name",
]

NULL_SENTINEL = {"nan", "none", "null", "n/a", "na", ""}

def null_mask(val):
    return not isinstance(val, str) or val.strip().lower() in NULL_SENTINEL

def make_actions(dataframe):
    for _, row in dataframe.iterrows():
        doc = {}
        for field in FIELDS:
            val = row.get(field)
            doc[field] = None if null_mask(val) else val.strip()

        # Build the completion payload for productName.suggest
        name = doc.get("productName")
        if name:
            doc["productName"] = {
                "input":   name,
                "suggest": {"input": [name]},
            }
            # streaming_bulk flattens _source, so keep name accessible
            # Revert: store as plain text — completion sub-field is defined in mapping
            doc["productName"] = name

        yield {
            "_index": INDEX_NAME,
            "_source": doc,
        }

# ── Ingest with progress bar ──────────────────────────────────────────────────
total_rows    = len(df)
success_total = 0
failed_total  = 0
errors_log    = []

start = time.time()

with tqdm(total=total_rows, desc="Indexing", unit="doc") as pbar:
    for ok, result in streaming_bulk(
        es,
        make_actions(df),
        chunk_size=CHUNK_SIZE,
        raise_on_error=False,
    ):
        if ok:
            success_total += 1
        else:
            failed_total += 1
            errors_log.append(result)
        pbar.update(1)

elapsed = time.time() - start

# ── Restore refresh interval + force merge ────────────────────────────────────
es.indices.put_settings(
    index=INDEX_NAME,
    body={"index": {"refresh_interval": "1s"}}
)
es.indices.refresh(index=INDEX_NAME)
es.indices.forcemerge(index=INDEX_NAME, max_num_segments=1)

# ── Summary ───────────────────────────────────────────────────────────────────
doc_count = es.count(index=INDEX_NAME)["count"]

print(f"✅ Ingest complete in {elapsed:.1f}s")
print(f"   Submitted : {total_rows:,}")
print(f"   Succeeded : {success_total:,}")
print(f"   Failed    : {failed_total:,}")
print(f"   Docs in index : {doc_count:,}")

if errors_log:
    print(f"\n⚠️  First 3 errors:")
    for e in errors_log[:3]:
        print(f"   {e}")


Indexing:   0%|          | 0/100000 [00:00<?, ?doc/s]

✅ Ingest complete in 41.7s
   Submitted : 100,000
   Succeeded : 100,000
   Failed    : 0
   Docs in index : 100,000


## Step 7 — Category Mapper: `predict_category()`
Given a product name, this function queries Elasticsearch and returns the **most likely category, sub-category and product category** together with a confidence score for each level.

**How it works:**
1. Run a 4-tier query (`match_phrase_prefix` → edge-ngram AND → edge-ngram OR → fuzzy) across `productName` + `productDescription`
2. Collect the top-N matching documents (default `top_n=50`)
3. For each category level, count how many of those docs belong to each value → frequency → **confidence %**
4. Return the winner at each level along with its score

| Field | Meaning |
|---|---|
| `category` | Broad category (e.g. *Sanitaryware & Bathroom Fittings*) |
| `sub_category` | Mid-level (e.g. *Bath Accessories*) |
| `product_category` | Most specific (e.g. *Bath Towels*) |
| `confidence` | % of top-N docs that agreed on this prediction |


In [ ]:
def predict_category(product_name: str, top_n: int = 50) -> dict:
    """
    Predict category / sub-category / product-category for a given product name.

    Uses a 4-tier relevance query and aggregates category frequencies over the
    top-N hits to compute per-level confidence scores.

    Parameters
    ----------
    product_name : str   — product name to classify
    top_n        : int   — how many top hits to consider (default 50)

    Returns
    -------
    dict with keys:
        query, total_hits, docs_used,
        category        { name, confidence, doc_count },
        sub_category    { name, confidence, doc_count },
        product_category{ name, confidence, doc_count },
        all_categories  { category: […], sub_category: […], product_category: […] }
    """
    q = product_name.strip()
    if not q:
        return {"error": "Empty product name"}

    resp = es.search(
        index=INDEX_NAME,
        size=0,   # we only need aggregations
        query={
            "bool": {
                "should": [
                    {"match_phrase_prefix": {"productName": {"query": q, "boost": 5.0}}},
                    {"match": {"productName.ngram": {"query": q, "operator": "and", "boost": 3.0}}},
                    {"match": {"productName.ngram": {"query": q, "operator": "or",  "boost": 1.0}}},
                    {"match": {"productName":       {"query": q, "fuzziness": "AUTO", "prefix_length": 1, "boost": 0.5}}},
                    {"match": {"productDescription":{"query": q, "operator": "or",   "boost": 0.3}}},
                ],
                "minimum_should_match": 1,
            }
        },
        aggs={
            # Use top-N hits via sampler, then break down by each category field
            "sample": {
                "sampler": {"shard_size": top_n},
                "aggs": {
                    "by_category": {
                        "terms": {"field": "category_name", "size": 10}
                    },
                    "by_sub_category": {
                        "terms": {"field": "subCategory_name", "size": 10}
                    },
                    "by_product_category": {
                        "terms": {"field": "productCategory_name", "size": 10}
                    },
                }
            }
        },
    )

    total_hits = resp["hits"]["total"]["value"]
    sample     = resp["aggregations"]["sample"]

    def parse_buckets(agg_key):
        buckets      = sample[agg_key]["buckets"]
        sum_other    = sample[agg_key].get("sum_other_doc_count", 0)
        total_in_agg = sum(b["doc_count"] for b in buckets) + sum_other
        if not buckets or total_in_agg == 0:
            return None, 0.0, 0, []
        top            = buckets[0]
        confidence_pct = round(top["doc_count"] / total_in_agg * 100, 1)
        all_items      = [
            {"name": b["key"], "doc_count": b["doc_count"],
             "confidence": round(b["doc_count"] / total_in_agg * 100, 1)}
            for b in buckets
        ]
        return top["key"], confidence_pct, top["doc_count"], all_items

    cat_name,  cat_conf,  cat_cnt,  cat_all  = parse_buckets("by_category")
    sub_name,  sub_conf,  sub_cnt,  sub_all  = parse_buckets("by_sub_category")
    prod_name, prod_conf, prod_cnt, prod_all = parse_buckets("by_product_category")

    # docs_used = total docs considered by the sampler
    docs_used = sum(b["doc_count"] for b in sample["by_category"]["buckets"]) \
                + sample["by_category"].get("sum_other_doc_count", 0)

    return {
        "query":       q,
        "total_hits":  total_hits,
        "docs_used":   docs_used,
        "category": {
            "name":       cat_name,
            "confidence": cat_conf,
            "doc_count":  cat_cnt,
        },
        "sub_category": {
            "name":       sub_name,
            "confidence": sub_conf,
            "doc_count":  sub_cnt,
        },
        "product_category": {
            "name":       prod_name,
            "confidence": prod_conf,
            "doc_count":  prod_cnt,
        },
        "all_categories": {
            "category":         cat_all,
            "sub_category":     sub_all,
            "product_category": prod_all,
        },
    }


# ── Quick smoke test ──────────────────────────────────────────────────────────
test_products = [
    "bath towels",
    "industrial pump",
    "led bulb 9w",
    "cotton saree",
    "cement 53 grade",
]

for name in test_products:
    r = predict_category(name)
    print(f"\n{'─'*60}")
    print(f"  Product  : {r['query']}")
    print(f"  Hits     : {r['total_hits']:,}  (sampled {r['docs_used']})")
    print(f"  Category        : {r['category']['name']}  ({r['category']['confidence']}%)")
    print(f"  Sub-Category    : {r['sub_category']['name']}  ({r['sub_category']['confidence']}%)")
    print(f"  Product Category: {r['product_category']['name']}  ({r['product_category']['confidence']}%)")
